In [1]:
!pip install librosa

import os
import numpy as np
import pandas as pd
import librosa
import librosa.display
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix

import tensorflow as tf
from tensorflow.keras import layers, models


In [2]:
from google.colab import files
files.upload()  # upload kaggle.json


Saving kaggle.json to kaggle.json


{'kaggle.json': b'{\r\n  "username": "YOUR_KAGGLE_USERNAME",\r\n  "key": "KGAT_2957138dac87ff071a98db5d2aad8c79"\r\n}'}

In [3]:
!pip install -q kaggle
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json


In [4]:
!kaggle datasets download -d bjoernjostein/the-circor-digiscope-phonocardiogram-dataset-v2 -p /content


Dataset URL: https://www.kaggle.com/datasets/bjoernjostein/the-circor-digiscope-phonocardiogram-dataset-v2
License(s): ODC Attribution License (ODC-By)
 87% 390M/449M [00:00<00:00, 539MB/s]
100% 449M/449M [00:00<00:00, 536MB/s]


In [5]:
!unzip -q /content/the-circor-digiscope-phonocardiogram-dataset-v2.zip -d /content/circor2022


In [6]:
csv_path = "/content/circor2022/training_data.csv"
data_path = "/content/circor2022/training_data/training_data"

df = pd.read_csv(csv_path)

# Keep only Present & Absent
df = df[df["Murmur"].isin(["Present", "Absent"])]

print(df["Murmur"].value_counts())


Murmur
Absent     695
Present    179
Name: count, dtype: int64


In [95]:
from scipy.signal import butter, lfilter

def bandpass_filter(audio, sr, lowcut=20, highcut=600, order=4):
    nyquist = 0.5 * sr
    low = lowcut / nyquist
    high = highcut / nyquist
    b, a = butter(order, [low, high], btype='band')
    filtered = lfilter(b, a, audio)
    return filtered


In [96]:
SR = 4000
DURATION = 10          # better than 5
SAMPLES = SR * DURATION
N_MELS = 64
MAX_LEN = 430          # adjust for 10 sec


In [97]:
def extract_features(file_path):
    try:
        audio, sr = librosa.load(file_path, sr=SR)

        # Fix length
        if len(audio) < SAMPLES:
            audio = np.pad(audio, (0, SAMPLES - len(audio)))
        else:
            audio = audio[:SAMPLES]

        # 🔥 Bandpass filter
        audio = bandpass_filter(audio, sr)

        # Optional harmonic separation
        harmonic, _ = librosa.effects.hpss(audio)
        audio = harmonic

        # Normalize
        audio = librosa.util.normalize(audio)

        # Mel Spectrogram
        mel = librosa.feature.melspectrogram(
            y=audio,
            sr=sr,
            n_mels=64,
            fmin=20,
            fmax=600
        )

        mel_db = librosa.power_to_db(mel)

        # Fix time dimension
        if mel_db.shape[1] < MAX_LEN:
            pad_width = MAX_LEN - mel_db.shape[1]
            mel_db = np.pad(mel_db, pad_width=((0,0),(0,pad_width)))
        else:
            mel_db = mel_db[:, :MAX_LEN]

        # Standardize
        mel_db = (mel_db - np.mean(mel_db)) / (np.std(mel_db) + 1e-6)

        return mel_db

    except:
        return None


In [98]:
# Unique patients
patients = df["Patient ID"].unique()

train_patients, test_patients = train_test_split(
    patients,
    test_size=0.2,
    random_state=42
)

train_df = df[df["Patient ID"].isin(train_patients)]
test_df = df[df["Patient ID"].isin(test_patients)]


In [99]:
def build_dataset(dataframe):
    X = []
    y = []

    all_files = os.listdir(data_path)

    for _, row in dataframe.iterrows():
        patient_id = str(row["Patient ID"]).strip()
        label = row["Murmur"]

        matched_files = [
            f for f in all_files
            if f.startswith(patient_id + "_") and f.endswith(".wav")
        ]

        for file in matched_files:
            file_path = os.path.join(data_path, file)
            features = extract_features(file_path)

            if features is not None:
                X.append(features)
                y.append(label)

    return np.array(X), np.array(y)


In [100]:
X_train, y_train = build_dataset(train_df)
X_test, y_test = build_dataset(test_df)

print("Train samples:", len(X_train))
print("Test samples:", len(X_test))


Train samples: 2403
Test samples: 604


In [101]:
print("Before reshape:", X_train.shape)

X_train = X_train[..., np.newaxis]
X_test = X_test[..., np.newaxis]

print("After reshape:", X_train.shape)


Before reshape: (2403, 64, 430)
After reshape: (2403, 64, 430, 1)


In [102]:
le = LabelEncoder()
y_train = le.fit_transform(y_train)
y_test = le.transform(y_test)

print("Label mapping:")
for i, label in enumerate(le.classes_):
    print(i, "=", label)


Label mapping:
0 = Absent
1 = Present


In [103]:
class_weights = {
    0: 1.0,
    1: 4.0
}

print(class_weights)


{0: 1.0, 1: 4.0}


In [104]:
model = tf.keras.Sequential([

    tf.keras.layers.Input(shape=X_train.shape[1:]),

    tf.keras.layers.Conv2D(32, (3,3), padding='same'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Activation('relu'),
    tf.keras.layers.MaxPooling2D((2,2)),

    tf.keras.layers.Conv2D(64, (3,3), padding='same'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Activation('relu'),
    tf.keras.layers.MaxPooling2D((2,2)),

    tf.keras.layers.Conv2D(128, (3,3), padding='same'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Activation('relu'),
    tf.keras.layers.MaxPooling2D((2,2)),

    tf.keras.layers.Dropout(0.4),

    tf.keras.layers.GlobalAveragePooling2D(),

    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dropout(0.5),

    tf.keras.layers.Dense(1, activation='sigmoid')
])


In [105]:
optimizer = tf.keras.optimizers.Adam(learning_rate=0.0005)

model.compile(
    optimizer=optimizer,
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.AUC()]
)

model.summary()


Model: "sequential_14"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_47 (Conv2D)              │ (None, 64, 430, 32)    │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_44          │ (None, 64, 430, 32)    │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_18 (Activation)      │ (None, 64, 430, 32)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_47 (MaxPooling2D) │ (None, 32, 215, 32)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_48 (Conv2D)              │ (None, 32, 215, 64)    │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_45          │ (None, 32, 215, 64)    │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_19 (Activation)      │ (None, 32, 215, 64)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_48 (MaxPooling2D) │ (None, 16, 107, 64)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_49 (Conv2D)              │ (None, 16, 107, 128)   │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_46          │ (None, 16, 107, 128)   │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_20 (Activation)      │ (None, 16, 107, 128)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_49 (MaxPooling2D) │ (None, 8, 53, 128)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_20 (Dropout)            │ (None, 8, 53, 128)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_6      │ (None, 128)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_28 (Dense)                │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_21 (Dropout)            │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_29 (Dense)                │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 101,889 (398.00 KB)

 Trainable params: 101,441 (396.25 KB)

 Non-trainable params: 448 (1.75 KB)

In [106]:
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=7,
    restore_best_weights=True
)

reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=3,
    min_lr=1e-6
)


In [107]:
history = model.fit(
    X_train,
    y_train,
    epochs=20,
    batch_size=16,
    validation_data=(X_test, y_test),
    class_weight=class_weights,
    callbacks=[early_stop, reduce_lr]
)


Epoch 1/20
151/151 ━━━━━━━━━━━━━━━━━━━━ 17s 58ms/step - accuracy: 0.5464 - auc_3: 0.5472 - loss: 1.1183 - val_accuracy: 0.2930 - val_auc_3: 0.7026 - val_loss: 0.7291 - learning_rate: 5.0000e-04
Epoch 2/20
151/151 ━━━━━━━━━━━━━━━━━━━━ 3s 21ms/step - accuracy: 0.5485 - auc_3: 0.6083 - loss: 1.0739 - val_accuracy: 0.5745 - val_auc_3: 0.7412 - val_loss: 0.6632 - learning_rate: 5.0000e-04
Epoch 3/20
151/151 ━━━━━━━━━━━━━━━━━━━━ 3s 20ms/step - accuracy: 0.6477 - auc_3: 0.6754 - loss: 1.0298 - val_accuracy: 0.5430 - val_auc_3: 0.7808 - val_loss: 0.6859 - learning_rate: 5.0000e-04
Epoch 4/20
151/151 ━━━━━━━━━━━━━━━━━━━━ 3s 20ms/step - accuracy: 0.7324 - auc_3: 0.7060 - loss: 0.9791 - val_accuracy: 0.3377 - val_auc_3: 0.7846 - val_loss: 0.8916 - learning_rate: 5.0000e-04
Epoch 5/20
151/151 ━━━━━━━━━━━━━━━━━━━━ 3s 21ms/step - accuracy: 0.7582 - auc_3: 0.7306 - loss: 0.9290 - val_accuracy: 0.7964 - val_auc_3: 0.6452 - val_loss: 0.4879 - learning_rate: 5.0000e-04
Epoch 6/20
151/151 ━━━━━━━━━━━━━━━

In [109]:
loss, accuracy, auc = model.evaluate(X_test, y_test)

print("Test Accuracy:", accuracy)
print("Test AUC:", auc)

y_pred = (model.predict(X_test) > 0.25).astype(int)

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))


19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.8073 - auc_3: 0.7592 - loss: 0.4535
Test Accuracy: 0.8460264801979065
Test AUC: 0.7910146713256836
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step

Confusion Matrix:
[[434  45]
 [ 55  70]]

Classification Report:
              precision    recall  f1-score   support

           0       0.89      0.91      0.90       479
           1       0.61      0.56      0.58       125

    accuracy                           0.83       604
   macro avg       0.75      0.73      0.74       604
weighted avg       0.83      0.83      0.83       604



In [111]:
model.save("heart_murmur_model.keras")

In [114]:
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]

tflite_model = converter.convert()

with open("heart_murmur_model_quantized.tflite", "wb") as f:
    f.write(tflite_model)


Saved artifact at '/tmp/tmpw94upw1y'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 64, 430, 1), dtype=tf.float32, name='keras_tensor_199')
Output Type:
  TensorSpec(shape=(None, 1), dtype=tf.float32, name=None)
Captures:
  140489832096464: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140489832091280: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140489832094928: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140489832094352: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140489832093584: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140489832092240: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140489832095120: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140489832092816: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140489832095312: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140489832093968: TensorSpec(shape=(), dtype=tf.resource, name=None)
  14048983208

In [115]:
files.download("heart_murmur_model_quantized.tflite")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>